In [33]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [34]:
import os
import shutil
import yaml

In [35]:
from ultralytics import YOLO

In [36]:
model = YOLO("yolov8n.pt")

In [37]:
SOURCE_DIR = "/kaggle/input/datasets/valentynsichkar/traffic-signs-dataset-in-yolo-format/ts/ts"
TRAIN_TXT  = "/kaggle/input/datasets/valentynsichkar/traffic-signs-dataset-in-yolo-format/train.txt"
TEST_TXT   = "/kaggle/input/datasets/valentynsichkar/traffic-signs-dataset-in-yolo-format/test.txt"

OUT_DATASET = "/kaggle/working/yolo_dataset"

TRAIN_IMG = f"{OUT_DATASET}/images/train"
VAL_IMG   = f"{OUT_DATASET}/images/val"
TRAIN_LAB = f"{OUT_DATASET}/labels/train"
VAL_LAB   = f"{OUT_DATASET}/labels/val"

for p in [TRAIN_IMG, VAL_IMG, TRAIN_LAB, VAL_LAB]:
    os.makedirs(p, exist_ok=True)

def copy_data(txt_file, img_out, lab_out):
    with open(txt_file, "r") as f:
        lines = f.readlines()

    for line in lines:
        img_name = os.path.basename(line.strip())
        label_name = img_name.replace(".jpg", ".txt")

        src_img = os.path.join(SOURCE_DIR, img_name)
        src_lab = os.path.join(SOURCE_DIR, label_name)

        if os.path.exists(src_img) and os.path.exists(src_lab):
            shutil.copy(src_img, os.path.join(img_out, img_name))
            shutil.copy(src_lab, os.path.join(lab_out, label_name))

copy_data(TRAIN_TXT, TRAIN_IMG, TRAIN_LAB)
copy_data(TEST_TXT, VAL_IMG, VAL_LAB)

print("Dataset successfully converted to YOLO format")

Dataset successfully converted to YOLO format


In [38]:
classes_path = "/kaggle/input/datasets/valentynsichkar/traffic-signs-dataset-in-yolo-format/classes.names"

with open(classes_path) as f:
    names = f.read().splitlines()

data = {
    "path": OUT_DATASET,
    "train": "images/train",
    "val": "images/val",
    "nc": len(names),
    "names": names
}

with open(f"{OUT_DATASET}/data.yaml", "w") as f:
    yaml.dump(data, f)

print("data.yaml created")

data.yaml created


In [39]:
model.train(data=f"{OUT_DATASET}/data.yaml", epochs=100, imgsz=640, batch=16)

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8c50bf5f70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [41]:
path = "/kaggle/working/runs/detect/train/weights/best.pt"

In [13]:
!pip install onnx onnxscript coremltools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 15.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 14.0 MB/s eta 0:00:00


In [42]:
import coremltools as ct
import onnx
from ultralytics import YOLO

In [31]:
!pip install coremltools


In [44]:
model = YOLO("/kaggle/working/runs/detect/train-2/weights/best.pt")

model.export(format="coreml", half=False,
    int8=False)

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/kaggle/working/runs/detect/train-2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (5.9 MB)

CoreML: starting export with coremltools 9.0...


Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '909', of the source model, has been renamed to 'var_909' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 120.28 passes/s]


CoreML: export success ✅ 5.0s, saved as '/kaggle/working/runs/detect/train-2/weights/best.mlpackage' (5.9 MB)

Export complete (5.3s)
Results saved to /kaggle/working/runs/detect/train-2/weights/best.mlpackage
Predict:         yolo predict task=detect model=/kaggle/working/runs/detect/train-2/weights/best.mlpackage imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/detect/train-2/weights/best.mlpackage imgsz=640 data=/kaggle/working/yolo_dataset/data.yaml  
Visualize:       https://netron.app


PosixPath('/kaggle/working/runs/detect/train-2/weights/best.mlpackage')

In [45]:
import shutil

path = "/kaggle/working/runs/detect/train-2/weights/best.mlpackage"

shutil.make_archive(
    "/kaggle/working/best_mlpackage",
    "zip",
    path
)

'/kaggle/working/best_mlpackage.zip'